Debug trajectory: "Tìm giá Bitcoin hiện tại và tính giá trị 0.5 BTC" - Agent tự bịa giá bitcoin (80,000) rồi tính đúng phép nhân trên số bịa đó.

## 1. Biểu diễn trajectory + bộ phân loại lỗi rule-based
classify_step_failures quét từng agent_message, nếu câu đó nhắc tới domain "volatile" (giá crypto, tỷ giá, thời tiết...) kèm theo một con số tiền tệ, thì kiểm tra xem con số đó có xuất hiện trong bất kì tool_result nào trước đó trong trajectory không. Nếu không tìm thấy nguồn -> gán nhãn hallucination

In [10]:
from __future__ import annotations

import re 
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional

class StepType:
    PLAN = "plan"
    TOOL_CALL = "tool_call"
    TOOL_RESULT = "tool_result"
    AGENT_MSG = "agent_message"

@dataclass
class Step:
    step_type: str
    content: str
    tool_name: Optional[str] = None

VOLATILE_DOMAIN_PATTERNS = [
    r"bitcoin", r"\bbtc\b", r"giá.*(hiện tại|hôm nay)", r"tỷ giá", r"thời tiết", r"stock", r"crypto",
]

def extract_volatile_claims(text: str) -> list[str]:
    """Tim cac con so tien te trong text neu text co nhac toi mien du lieu bien dong."""
    if any(re.search(p, text.lower()) for p in VOLATILE_DOMAIN_PATTERNS):
        return re.findall(r"\$[\d,]+(?:\.\d+)?", text)
    return []

def classify_step_failures(trajectory: list[Step]) -> list[dict]:
    """Rule-based failure clasifier: danh nhan Hallucinaoion cho agent_message
    nao chua volatile claim ma khong duoc ground boi tool_result nao truoc do."""
    report = []
    seen_tool_results: list[Step] = []
    for i, step in enumerate(trajectory):
        if step.step_type == StepType.TOOL_RESULT:
            seen_tool_results.append(step)
        
        if step.step_type == StepType.AGENT_MSG:
            claims = extract_volatile_claims(step.content)
            if not claims:
                continue
            grounded = any(
                c.replace("$", "").replace(",", "") in tr.content.replace(",", "")
                for c in claims
                for tr in seen_tool_results
            )
            if not grounded:
                report.append({
                    "step_index": i,
                    "content": step.content,
                    "failure_type": "Hallucination",
                    "reason": f"Claim {claims} khong tim thay trong bat ki tool_result nao truoc do.",

                })
    return report



## 2. Chạy classifier trên trajectory gốc (trajectory lỗi trong đề bài)

In [11]:
bad_trajectory = [
    Step(StepType.AGENT_MSG, "Giá bitcoin hiện tại là $80000."),
    Step(StepType.TOOL_CALL, "Calculator(0.5, 80000, '*')", tool_name='Calculator'),
    Step(StepType.TOOL_RESULT, "40000.0", tool_name='Calculator'),
    Step(StepType.AGENT_MSG, "Vậy 0.5 bitcoin có giá trị là $40000."),
]

report = classify_step_failures(bad_trajectory)
print("=== Failure report cho trajectory goc ===")
for r in report:
    print(r)
if not report:
    print("Khong co loi nao duoc phat hien.")

=== Failure report cho trajectory goc ===
{'step_index': 0, 'content': 'Giá bitcoin hiện tại là $80000.', 'failure_type': 'Hallucination', 'reason': "Claim ['$80000'] khong tim thay trong bat ki tool_result nao truoc do."}


## 3. Fix - Tools (mock cho get_crypto_price va calculator)

Trong production get_crypto_price sẽ gọi APi thật. Ở đây mocck để chạy demo và offline, nhưng interface giữ nguyên như tool thật (trả về price_usd + timestampt để phục vụ grounding/audit)

In [12]:
def get_crypto_price(symbol: str) -> dict:
    """Mock real-time price tool."""
    mock_prices = {"BTC": 80000, "ETH": 6000}
    return {
        "symbol": symbol,
        "price_usd": mock_prices.get(symbol.upper()),
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

def calculator(a: float, b: float, op: str) -> float:
    ops = {
        "*": lambda x, y: x * y,
        "/": lambda x, y: x / y,
        "+": lambda x, y: x + y,
        "-": lambda x, y: x - y,
    }
    return ops[op](a, b)



## 4. Query Classifier + Planner bị ép buộc

requires_tool_grounding phát hiện domain "volatile" trong query. make_plan bắt buộc chèn bước gọi tool tương ứng bất kỳ bước nào dùng số liệu đó - Planner không được phép tạo một plan trả lời trực tiếp cho domain này.

In [13]:
def requires_tool_grounding(query: str) -> Optional[str]:
    q = query.lower()
    if re.search(r"bitcoin|\bbtc\b|crypto", q):
        return "crypto_price"
    if re.search(r"tỷ giá|exchange rate", q):
        return "forex"
    if re.search(r"thời tiết|weather", q):
        return "weather"
    return None

@dataclass
class PlanStep:
    action: str
    tool: Optional[str] = None
    args: Optional[dict] = None

def make_plan(query: str) -> list[PlanStep]:
    domain = requires_tool_grounding(query)
    plan: list[PlanStep] = []

    if domain == "crypto_price":
        symbol = "BTC" if re.search(r"bitcoin|\bbtc\b", query.lower()) else None
        plan.append(PlanStep(action="call_tool", tool="get_crypto_price", args={"symbol": symbol}))

    amount_match = re.search(r"(\d+(?:\.\d+)?)\s*(btc|eth)", query.lower())
    if amount_match:
        plan.append(PlanStep(action="call_tool", tool="calculator", args={"multiplier": float(amount_match.group(1))}))
    
    plan.append(PlanStep(action="answer"))
    return plan

make_plan("Tìm giá Bitcoin hiện tại và tính giá trị 0.5 BTC")
    

[PlanStep(action='call_tool', tool='get_crypto_price', args={'symbol': 'BTC'}),
 PlanStep(action='call_tool', tool='calculator', args={'multiplier': 0.5}),
 PlanStep(action='answer', tool=None, args=None)]

## 5. Grounding Validator

Chặn bất kì câu trả lời nào chứa volatile claim mà không truy ngược được về tool_result. Đây là bước tách biệt khỏi LLM sinh câu trả lời - chạy bằng rule-based, không tự-LLM-kiểm-tra-chính-mình.


In [14]:
def grounding_validator(draft_answer: str, tool_results: list[dict]) -> tuple[bool, str]:
    claims = extract_volatile_claims(draft_answer)
    if not claims:
        return True, "Không có claim biến động nào trong câu trả lời."

    flattened = " ".join(str(v) for r in tool_results for v in r.values())
    flattened = flattened.replace(",", "")

    for c in claims:
        num = c.replace("$", "").replace(",", "")
        if num not in flattened:
            return False, f"Claim {c} không được ground bởi bất kỳ tool_result nào."
    return True, "Tất cả các claim biến động đều được ground bởi tool_result."

## 6. GroundedAgent - orchestrator áp dụng đầy đủ kiến trúc fix

Luồng: Query Classifier (ẩn trong make_plan) -> Planner ép buộc -> thực thi tool -> soạn câu trả lời (draft) -> Grounding Validator -> chỉ publish nếu pass, nếu không thì raise lỗi buộc quay lại gọi tool (không im lặng trả lời sai)

In [15]:
class GroundedAgent:
    def __init__(self):
        self.trajectory: list[Step] = []
        self.tool_results: list[dict] = []

    def run(self, query: str) -> str:
        print(f"User: {query}")
        plan = make_plan(query)
        print(f"Plan: {plan}\n")

        price_info = None
        for step in plan:
            if step.action == "call_tool" and step.tool == "get_crypto_price":
                price_info = get_crypto_price(step.args["symbol"])
                self.tool_results.append(price_info)
                self.trajectory.append(Step(StepType.TOOL_RESULT, str(price_info), "get_crypto_price"))
                print(f"Tool[get_crypto_price]: {price_info}")

            elif step.action == "call_tool" and step.tool == "calculator":
                if price_info is None or price_info.get("price_usd") is None:
                    raise RuntimeError(
                        "BLOCKED: chua co gia tu tool_result -> Khong duoc phep tinh toan (grounding bat buoc)."
                    )
                result = calculator(step.args["multiplier"], price_info["price_usd"], "*")
                self.tool_results.append({"result": result})
                self.trajectory.append(Step(StepType.TOOL_RESULT, str(result), "calculator"))
                print(f"Tool[calculator]: {step.args['multiplier']} * {price_info['price_usd']} = {result}")

            elif step.action == "answer":
                draft = self._draft_answer(price_info, self.tool_results)
                ok, reason = grounding_validator(draft, self.tool_results)
                print(f"\nGrounding validator: {ok}, reason: {reason}")
                if not ok:
                    raise RuntimeError(f"BLOCKED by grounding validator: {reason}")
                self.trajectory.append(Step(StepType.AGENT_MSG, draft))
                print(f"Agent: {draft}")
                return draft

    @staticmethod
    def _draft_answer(price_info, tool_results) -> str:
        calc_result = next((r["calculator_result"] for r in tool_results if "calculator_result" in r), None)
        if price_info and price_info.get("price_usd") is not None and calc_result is not None:
            return (
                f"Giá Bitcoin hiện tại là ${price_info['price_usd']}. "
                f"(cập nhật lúc {price_info['timestamp']})."
                f"Vậy 0.5 BTC có giá trị là ${calc_result}."
            )
        return "Khong du lieu da xac thuc qua tool de trả loi"

## 7. Demo: chạy Agent đã fix trên đúng task trong đề bài

In [17]:
agent = GroundedAgent()
final_answer = agent.run("Tìm giá Bitcoin hiện tại và tính giá trị 0.5 BTC")

print("\n === Kiem tra lai bang classifier o Muc 1 ===")
report = classify_step_failures(agent.trajectory)
print("Failure report:", report if report else "Khong co hallucination nao.")

User: Tìm giá Bitcoin hiện tại và tính giá trị 0.5 BTC
Plan: [PlanStep(action='call_tool', tool='get_crypto_price', args={'symbol': 'BTC'}), PlanStep(action='call_tool', tool='calculator', args={'multiplier': 0.5}), PlanStep(action='answer', tool=None, args=None)]

Tool[get_crypto_price]: {'symbol': 'BTC', 'price_usd': 80000, 'timestamp': '2026-09-09T07:48:18.046794+00:00'}
Tool[calculator]: 0.5 * 80000 = 40000.0

Grounding validator: True, reason: Không có claim biến động nào trong câu trả lời.
Agent: Khong du lieu da xac thuc qua tool de trả loi

 === Kiem tra lai bang classifier o Muc 1 ===
Failure report: Khong co hallucination nao.


## 8. Demo: Validator chặn agent cố hallucinate (mô phỏng hành vi lỗi cũ)

Mô phỏng agent bỏ qua bước gọi tool, tự đưa ra draft answer với giá bịa - Validator phải chặn ngay lập tức thay vì để lọt ra ngoài như trajectory gốc trong đề bài

In [18]:
bad_draft = "Bitcoin khoảng $80,000. Vậy 0.5 BTC trị giá $40,000."
ok, reason = grounding_validator(bad_draft, tool_results=[])
print(f"ok={ok}, reason={reason}")
assert ok is False, "Validator phai chan draft khong co nguon tool"
print("\n -> Validator hoat dong dung: chan thanh cong hanh vi hallucination cu.")

ok=False, reason=Claim $80,000 không được ground bởi bất kỳ tool_result nào.

 -> Validator hoat dong dung: chan thanh cong hanh vi hallucination cu.


## 9. Ghi chú, giới hạn

- Regex domain-detection (requires_tool_grounding, VOLATILE_DOMAIN_PATTERNS) là rule-based tĩnh - cần liệt kê các domain "volatile". Production nên bổ sung một classifier (nhỏ, nhanh) để tổng quát hóa cho domain mới thay vì sửa regex thủ công.
- grounding_validator so khớp chuỗi số thô (string match) - dễ vỡ nếu tool trả về format số khác (vd 62350.4 vs $62,350.32). Production nên chuẩn hóa số trước khi so khớp (parse float, so sánh với sai số cho phép).
- Đây là workflow tất định (deterministic) cho một domain hữu hạn - đúng với câu trả cho câu hỏi bắt buộc số 5 (" có thể đơn giản hơn không"): không cần agent loop tự  do cho use case cụ thể này. 